In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import re
from collections import Counter, defaultdict
import sys 


In [23]:
class GraphCSVAnalyzer:
    def __init__(self, csv_folder="graph_exports"):
        self.csv_folder = csv_folder
        self.graphs = {}
        
    def load_graph_from_csvs(self, graph_name, patterns_csv, nodes_csv, relationships_csv):
        try:
            patterns_df = pd.read_csv(patterns_csv)
            nodes_df = pd.read_csv(nodes_csv)
            relationships_df = pd.read_csv(relationships_csv)
            
            self.graphs[graph_name] = {
                'patterns': patterns_df,
                'nodes': nodes_df,
                'relationships': relationships_df,
                'loaded_at': datetime.now()
            }
            
            print(f"Loaded {graph_name} successfully")
            return True
            
        except Exception as e:
            print(f"Error loading {graph_name}: {e}")
            return False
    
    def quick_stats(self, graph_name):
        if graph_name not in self.graphs:
            print(f"Graph {graph_name} not loaded")
            return
            
        graph = self.graphs[graph_name]
        patterns = graph['patterns']
        nodes = graph['nodes']
        relationships = graph['relationships']
        
        total_nodes = nodes['Count'].sum()
        total_relationships = relationships['Count'].sum()
        unique_patterns = len(patterns)
        unique_node_types = len(nodes)
        unique_rel_types = len(relationships)
        
        print(f"\nSTATS for {graph_name}:")
        print(f"Total Nodes: {total_nodes}")
        print(f"Total Relationships: {total_relationships}")
        print(f"Unique Node Types: {unique_node_types}")
        print(f"Unique Relationship Types: {unique_rel_types}")
        print(f"Unique Patterns: {unique_patterns}")
        
        return {
            'total_nodes': total_nodes,
            'total_relationships': total_relationships,
            'unique_node_types': unique_node_types,
            'unique_rel_types': unique_rel_types,
            'unique_patterns': unique_patterns
        }
    
    def compare_two_graphs(self, graph1_name, graph2_name):
        if graph1_name not in self.graphs or graph2_name not in self.graphs:
            print("One or both graphs not loaded")
            return
        
        g1 = self.graphs[graph1_name]
        g2 = self.graphs[graph2_name]
        
        patterns1 = set(zip(g1['patterns']['Source'], g1['patterns']['Relationship'], g1['patterns']['Target']))
        patterns2 = set(zip(g2['patterns']['Source'], g2['patterns']['Relationship'], g2['patterns']['Target']))
        
        pattern_match = len(patterns1 & patterns2) / len(patterns1 | patterns2) * 100
        
        nodes1 = set(g1['nodes']['NodeType'])
        nodes2 = set(g2['nodes']['NodeType'])
        
        node_match = len(nodes1 & nodes2) / len(nodes1 | nodes2) * 100
        
        rels1 = set(g1['relationships']['RelationshipType'])
        rels2 = set(g2['relationships']['RelationshipType'])
        
        rel_match = len(rels1 & rels2) / len(rels1 | rels2) * 100
        
        overall_score = (pattern_match * 0.5 + node_match * 0.3 + rel_match * 0.2)
        
        print(f"\nCOMPARISON: {graph1_name} vs {graph2_name}")
        print(f"Pattern Match: {pattern_match:.1f}%")
        print(f"Node Type Match: {node_match:.1f}%")
        print(f"Relationship Type Match: {rel_match:.1f}%")
        print(f"Overall Score: {overall_score:.1f}%")
        
        return {
            'pattern_match': pattern_match,
            'node_match': node_match,
            'rel_match': rel_match,
            'overall_score': overall_score
        }
    
    def find_missing_schema_elements_no_penalty(self, graph_name, expected_nodes, expected_relationships):
        if graph_name not in self.graphs:
            print(f"Graph {graph_name} not loaded")
            return
        
        graph = self.graphs[graph_name]
        actual_nodes = set(graph['nodes']['NodeType'])
        actual_rels = set(graph['relationships']['RelationshipType'])
        
        missing_nodes = set(expected_nodes) - actual_nodes
        missing_rels = set(expected_relationships) - actual_rels
        
        print(f"\nSCHEMA COMPLIANCE for {graph_name} (no extra penalty):")
        print(f"Missing Nodes: {list(missing_nodes) if missing_nodes else 'None'}")
        print(f"Missing Relationships: {list(missing_rels) if missing_rels else 'None'}")
        
        # Compliance only considers missing elements
        node_compliance = len(set(expected_nodes) & actual_nodes) / len(expected_nodes) * 100
        rel_compliance = len(set(expected_relationships) & actual_rels) / len(expected_relationships) * 100
        compliance_score = (node_compliance + rel_compliance) / 2  # simple average
        
        print(f"Schema Compliance Score: {compliance_score:.1f}%")
        
        return {
            'missing_nodes': list(missing_nodes),
            'missing_rels': list(missing_rels),
            'compliance_score': compliance_score
        }

    
    def export_comparison_report(self, graph1_name, graph2_name, filename=None):
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"graph_comparison_{graph1_name}_vs_{graph2_name}_{timestamp}.csv"
        
        comparison = self.compare_two_graphs(graph1_name, graph2_name)
        stats1 = self.quick_stats(graph1_name)
        stats2 = self.quick_stats(graph2_name)
        
        report_data = {
            'Metric': ['Total Nodes', 'Total Relationships', 'Unique Node Types', 
                      'Unique Relationship Types', 'Unique Patterns'],
            f'{graph1_name}': [stats1['total_nodes'], stats1['total_relationships'], 
                              stats1['unique_node_types'], stats1['unique_rel_types'], 
                              stats1['unique_patterns']],
            f'{graph2_name}': [stats2['total_nodes'], stats2['total_relationships'], 
                              stats2['unique_node_types'], stats2['unique_rel_types'], 
                              stats2['unique_patterns']]
        }
        
        report_df = pd.DataFrame(report_data)
        report_df.to_csv(filename, index=False)
        print(f"Comparison report saved to: {filename}")


if __name__ == "__main__":
    analyzer = GraphCSVAnalyzer()
    
    expected_nodes = ['Scrum', 'ScrumRole', 'ScrumArtifact', 'ScrumEvent', 'ScrumValue', 'Methodology', 'Process', 'Concept']
    expected_relationships = ['MENTIONS_ROLE', 'DESCRIBES_ARTIFACT', 'EXPLAINS_EVENT', 'PROMOTES_VALUE', 
                            'USES_METHODOLOGY', 'INCLUDES_PROCESS', 'DEFINES_CONCEPT', 'HAS_RESPONSIBILITY',
                            'CREATES_ARTIFACT', 'PARTICIPATES_IN', 'PRODUCES_ARTIFACT', 'SUPPORTS_VALUE', 'RELATES_TO']
    
    analyzer.load_graph_from_csvs("EBM", "graph_exports/relationship_patterns_EBM_2025-9-17.csv", "graph_exports/node_distribution_EBM_2025-9-17.csv", "graph_exports/relationship_frequency_EBM_2025-9-17.csv")
    analyzer.load_graph_from_csvs("Nexus", "graph_exports/relationship_patterns_nexus_2025-9-17.csv", "graph_exports/node_distribution_nexus_2025-9-17.csv", "graph_exports/relationship_frequency_nexus_2025-9-17.csv")
    
    # Compare graphs
    analyzer.compare_two_graphs("EBM", "Nexus")
    analyzer.find_missing_schema_elements_no_penalty("EBM", expected_nodes, expected_relationships)
    analyzer.find_missing_schema_elements_no_penalty("Nexus",expected_nodes, expected_relationships)
    analyzer.export_comparison_report("EBM", "Nexus")

    


Loaded EBM successfully
Loaded Nexus successfully

COMPARISON: EBM vs Nexus
Pattern Match: 3.7%
Node Type Match: 90.0%
Relationship Type Match: 10.5%
Overall Score: 30.9%

SCHEMA COMPLIANCE for EBM (no extra penalty):
Missing Nodes: ['ScrumEvent', 'ScrumArtifact', 'Concept', 'ScrumValue', 'Process', 'Methodology', 'Scrum', 'ScrumRole']
Missing Relationships: ['PROMOTES_VALUE', 'INCLUDES_PROCESS', 'PARTICIPATES_IN', 'CREATES_ARTIFACT', 'HAS_RESPONSIBILITY', 'PRODUCES_ARTIFACT', 'SUPPORTS_VALUE', 'USES_METHODOLOGY', 'DESCRIBES_ARTIFACT', 'DEFINES_CONCEPT', 'MENTIONS_ROLE', 'EXPLAINS_EVENT']
Schema Compliance Score: 3.8%

SCHEMA COMPLIANCE for Nexus (no extra penalty):
Missing Nodes: ['ScrumEvent', 'ScrumArtifact', 'Concept', 'ScrumValue', 'Process', 'Methodology', 'Scrum', 'ScrumRole']
Missing Relationships: ['PROMOTES_VALUE', 'INCLUDES_PROCESS', 'CREATES_ARTIFACT', 'PRODUCES_ARTIFACT', 'HAS_RESPONSIBILITY', 'SUPPORTS_VALUE', 'USES_METHODOLOGY', 'DESCRIBES_ARTIFACT', 'DEFINES_CONCEPT', '

In [24]:
class KnowledgeGraphTripleAnalyzer:
    def __init__(self, file1_path=None, file2_path=None, graph1_name="Graph1", graph2_name="Graph2"):
        self.graph1_triples = []
        self.graph2_triples = []
        self.graph1_name = graph1_name
        self.graph2_name = graph2_name

        if file1_path and file2_path:
            self.graph1_triples = self.parse_triple_file(file1_path)
            print(f"Loaded {len(self.graph1_triples)} triples from {file1_path}")
            self.graph2_triples = self.parse_triple_file(file2_path)
            print(f"Loaded {len(self.graph2_triples)} triples from {file2_path}")

    def parse_triple_file(self, file_path):
        triples = []
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        pattern = r'\(([^)]+)\)\s*--\[([^\]]+)\]-->\s*\(([^)]+)\)'
        matches = re.findall(pattern, content)
        for match in matches:
            triples.append((match[0].strip(), match[1].strip(), match[2].strip()))
        return triples
    
    def load_graphs(self, file1_path, file2_path, graph1_name="Graph1", graph2_name="Graph2"):
        self.graph1_name = graph1_name
        self.graph2_name = graph2_name
        
        print(f"Loading {graph1_name} from {file1_path}...")
        self.graph1_triples = self.parse_triple_file(file1_path)
        print(f"Loaded {len(self.graph1_triples)} triples")
        
        print(f"Loading {graph2_name} from {file2_path}...")
        self.graph2_triples = self.parse_triple_file(file2_path)
        print(f"Loaded {len(self.graph2_triples)} triples")
    
    def analyze_triple_overlap(self):
        set1 = set(self.graph1_triples)
        set2 = set(self.graph2_triples)
        
        common_triples = set1 & set2
        unique_to_graph1 = set1 - set2
        unique_to_graph2 = set2 - set1
        total_unique_triples = len(set1 | set2)
        
        triple_overlap = len(common_triples) / total_unique_triples * 100 if total_unique_triples > 0 else 0
        
        print(f"\n" + "="*70)
        print("TRIPLE OVERLAP ANALYSIS")
        print("="*70)
        print(f"{self.graph1_name}: {len(set1)} unique triples")
        print(f"{self.graph2_name}: {len(set2)} unique triples")
        print(f"Common triples: {len(common_triples)}")
        print(f"Unique to {self.graph1_name}: {len(unique_to_graph1)}")
        print(f"Unique to {self.graph2_name}: {len(unique_to_graph2)}")
        print(f"Total unique triples: {total_unique_triples}")
        print(f"Triple Overlap: {triple_overlap:.1f}%")
        
        if common_triples:
            print(f"\nSample Common Triples (showing first 5):")
            for i, triple in enumerate(sorted(common_triples)[:5]):
                print(f"  {i+1}. ({triple[0]}) --[{triple[1]}]--> ({triple[2]})")
        
        return {
            'common_triples': common_triples,
            'unique_to_graph1': unique_to_graph1,
            'unique_to_graph2': unique_to_graph2,
            'triple_overlap_percent': triple_overlap,
            'total_triples_graph1': len(set1),
            'total_triples_graph2': len(set2)
        }
    
    def analyze_predicate_overlap(self):
        predicates1 = [triple[1] for triple in self.graph1_triples]
        predicates2 = [triple[1] for triple in self.graph2_triples]
        
        pred_set1 = set(predicates1)
        pred_set2 = set(predicates2)
        
        common_preds = pred_set1 & pred_set2
        unique_preds1 = pred_set1 - pred_set2
        unique_preds2 = pred_set2 - pred_set1
        total_unique_preds = len(pred_set1 | pred_set2)
        
        pred_overlap = len(common_preds) / total_unique_preds * 100 if total_unique_preds > 0 else 0
        
        pred_count1 = Counter(predicates1)
        pred_count2 = Counter(predicates2)
        
        print(f"\n" + "="*70)
        print("PREDICATE (RELATIONSHIP) ANALYSIS")
        print("="*70)
        print(f"{self.graph1_name}: {len(pred_set1)} unique predicates")
        print(f"{self.graph2_name}: {len(pred_set2)} unique predicates")
        print(f"Common predicates: {len(common_preds)}")
        print(f"Predicate Overlap: {pred_overlap:.1f}%")
        
        if common_preds:
            print(f"\nCommon Predicates (with frequencies):")
            common_sorted = sorted([(pred, pred_count1[pred], pred_count2[pred]) 
                                  for pred in common_preds], 
                                 key=lambda x: x[1] + x[2], reverse=True)
            for pred, count1, count2 in common_sorted[:10]:
                print(f"  {pred}: {count1} vs {count2}")
        
        if unique_preds1:
            print(f"\nTop Predicates Unique to {self.graph1_name}:")
            unique_sorted1 = sorted([(pred, pred_count1[pred]) for pred in unique_preds1], 
                                  key=lambda x: x[1], reverse=True)
            for pred, count in unique_sorted1[:5]:
                print(f"  {pred} ({count})")
        
        if unique_preds2:
            print(f"\nTop Predicates Unique to {self.graph2_name}:")
            unique_sorted2 = sorted([(pred, pred_count2[pred]) for pred in unique_preds2], 
                                  key=lambda x: x[1], reverse=True)
            for pred, count in unique_sorted2[:5]:
                print(f"  {pred} ({count})")
        
        return {
            'common_predicates': common_preds,
            'predicate_overlap_percent': pred_overlap,
            'unique_to_graph1': unique_preds1,
            'unique_to_graph2': unique_preds2,
            'predicate_frequencies': {
                'graph1': pred_count1,
                'graph2': pred_count2
            }
        }
    
    def analyze_entity_overlap(self):
        entities1 = set()
        entities2 = set()
        
        for triple in self.graph1_triples:
            entities1.add(triple[0]) 
            entities1.add(triple[2])  
            
        for triple in self.graph2_triples:
            entities2.add(triple[0])  
            entities2.add(triple[2])  
        
        common_entities = entities1 & entities2
        unique_entities1 = entities1 - entities2
        unique_entities2 = entities2 - entities1
        total_unique_entities = len(entities1 | entities2)
        
        entity_overlap = len(common_entities) / total_unique_entities * 100 if total_unique_entities > 0 else 0
        
        print(f"\n" + "="*70)
        print("ENTITY OVERLAP ANALYSIS")
        print("="*70)
        print(f"{self.graph1_name}: {len(entities1)} unique entities")
        print(f"{self.graph2_name}: {len(entities2)} unique entities")
        print(f"Common entities: {len(common_entities)}")
        print(f"Entity Overlap: {entity_overlap:.1f}%")
        
        if common_entities:
            print(f"\nSample Common Entities:")
            for i, entity in enumerate(sorted(common_entities)[:10]):
                print(f"  {i+1}. {entity}")
        
        return {
            'common_entities': common_entities,
            'entity_overlap_percent': entity_overlap,
            'unique_to_graph1': unique_entities1,
            'unique_to_graph2': unique_entities2,
            'total_entities_graph1': len(entities1),
            'total_entities_graph2': len(entities2)
        }
    
    def analyze_structural_patterns(self):
        subj_pred1 = set((triple[0], triple[1]) for triple in self.graph1_triples)
        subj_pred2 = set((triple[0], triple[1]) for triple in self.graph2_triples)
        
        common_subj_pred = subj_pred1 & subj_pred2
        subj_pred_overlap = len(common_subj_pred) / len(subj_pred1 | subj_pred2) * 100 if (subj_pred1 | subj_pred2) else 0
        
        pred_obj1 = set((triple[1], triple[2]) for triple in self.graph1_triples)
        pred_obj2 = set((triple[1], triple[2]) for triple in self.graph2_triples)
        
        common_pred_obj = pred_obj1 & pred_obj2
        pred_obj_overlap = len(common_pred_obj) / len(pred_obj1 | pred_obj2) * 100 if (pred_obj1 | pred_obj2) else 0
        
        print(f"\n" + "="*70)
        print("STRUCTURAL PATTERN ANALYSIS")
        print("="*70)
        print(f"Subject-Predicate Pattern Overlap: {subj_pred_overlap:.1f}%")
        print(f"Predicate-Object Pattern Overlap: {pred_obj_overlap:.1f}%")
        
        return {
            'subj_pred_overlap': subj_pred_overlap,
            'pred_obj_overlap': pred_obj_overlap
        }
    
    def generate_comprehensive_report(self):
        triple_analysis = self.analyze_triple_overlap()
        predicate_analysis = self.analyze_predicate_overlap()
        entity_analysis = self.analyze_entity_overlap()
        pattern_analysis = self.analyze_structural_patterns()
        
        overall_score = (
            triple_analysis['triple_overlap_percent'] * 0.4 +      
            predicate_analysis['predicate_overlap_percent'] * 0.3 + 
            entity_analysis['entity_overlap_percent'] * 0.2 +       
            (pattern_analysis['subj_pred_overlap'] + pattern_analysis['pred_obj_overlap']) / 2 * 0.1  
        )
        
        print(f"\n" + "="*70)
        print("COMPREHENSIVE SIMILARITY REPORT")
        print("="*70)
        print(f"Triple Overlap: {triple_analysis['triple_overlap_percent']:.1f}%")
        print(f"Predicate Overlap: {predicate_analysis['predicate_overlap_percent']:.1f}%")
        print(f"Entity Overlap: {entity_analysis['entity_overlap_percent']:.1f}%")
        print(f"Structural Pattern Overlap: {(pattern_analysis['subj_pred_overlap'] + pattern_analysis['pred_obj_overlap']) / 2:.1f}%")
        print(f"\nOVERALL SIMILARITY SCORE: {overall_score:.1f}%")
        
        if overall_score >= 70:
            quality = "EXCELLENT - Very similar knowledge extraction"
        elif overall_score >= 50:
            quality = "GOOD - Reasonable similarity with some differences"
        elif overall_score >= 30:
            quality = "MODERATE - Some overlap but significant differences"
        else:
            quality = "LOW - Substantial differences in knowledge extraction"
        
        print(f"Quality Assessment: {quality}")
        
        return {
            'triple_overlap': triple_analysis['triple_overlap_percent'],
            'predicate_overlap': predicate_analysis['predicate_overlap_percent'],
            'entity_overlap': entity_analysis['entity_overlap_percent'],
            'structural_overlap': (pattern_analysis['subj_pred_overlap'] + pattern_analysis['pred_obj_overlap']) / 2,
            'overall_score': overall_score,
            'quality_assessment': quality
        }


if __name__ == "__main__":
    analyzer = KnowledgeGraphTripleAnalyzer(
        "Triples and Semantic Blocks/documentEBM_triples_v1.txt",
        "Triples and Semantic Blocks/documentNexusGuideScrum_triples_v1.txt",
        "Method1",
        "Method2")
    analyzer.generate_comprehensive_report()
    print("Knowledge Graph Triple Analyzer Ready!")
    print("Usage:")
    
    report_filename = ("type2Comparison_Itext2kg_EBM_vs_Nexus")
    

    with open(report_filename, "w", encoding="utf-8") as f:
        original_stdout = sys.stdout
        sys.stdout = f
        analyzer.generate_comprehensive_report()
        sys.stdout = original_stdout

    print(f"Comparison report saved to: {report_filename}")


Loaded 198 triples from Triples and Semantic Blocks/documentEBM_triples_v1.txt
Loaded 202 triples from Triples and Semantic Blocks/documentNexusGuideScrum_triples_v1.txt

TRIPLE OVERLAP ANALYSIS
Method1: 198 unique triples
Method2: 202 unique triples
Common triples: 3
Unique to Method1: 195
Unique to Method2: 199
Total unique triples: 397
Triple Overlap: 0.8%

Sample Common Triples (showing first 5):
  1. (Sprint Planning) --[related_to]--> (Daily Scrum)
  2. (Sprint Review) --[related_to]--> (Sprint Retrospective)
  3. (Subject) --[Predicate]--> (Object)

PREDICATE (RELATIONSHIP) ANALYSIS
Method1: 129 unique predicates
Method2: 116 unique predicates
Common predicates: 24
Predicate Overlap: 10.9%

Common Predicates (with frequencies):
  related_to: 14 vs 12
  works_on: 7 vs 9
  is_part_of: 4 vs 8
  include: 6 vs 6
  involves: 6 vs 5
  relates_to: 8 vs 2
  uses: 6 vs 4
  is_related_to: 5 vs 4
  defines: 4 vs 5
  manages: 1 vs 6

Top Predicates Unique to Method1:
  is_founder (2)
  is_me